In [1]:
import os
from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

In [ ]:
SYSTEM_PROMPT_STRING = """
Your task is to partition the 16 words into 4 groups of 4 words based on shared connections. 
Output requirements (STRICT): 
You will be given 16 words.
OUTPUT ONLY the final groups of words.
Do NOT provide reasoning or explanations under any circumstances.
DO NOT output any text other than the 4 groups.
Make sure there are EXACTLY 4 groups of 4 words each with their category names. NO EXCEPTIONS.
Return the answer exactly in this format:

GROUP 1: word1 || word2 || word3 || word4
GROUP 2: word1 || word2 || word3 || word4
GROUP 3: word1 || word2 || word3 || word4
GROUP 4: word1 || word2 || word3 || word4
"""

In [ ]:
import re

def call_llm(system_prompt, user_prompt,
                        model = "llama3.1-8b",
                        temperature = 0.1,
                        max_tokens = 600):
    """
    Sends a chat request to the Cerebras API and returns the response content.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        msg = response.choices[0].message
        return (msg.content or "").strip()
    
    except Exception as e:
        return f"ERROR: {e}"

def convert_puzzle_to_prompt(words16):
    word_list_str = ", ".join(words16)
    return f"Here are the 16 words: {word_list_str}"

def parse_response_to_pred_groups(response):
    pattern = r"GROUP \d+: (.+)"
    
    groups = re.findall(pattern, response)
    parsed_groups = [ [word.strip() for word in group.split("||")] for group in groups ]
    
    return parsed_groups

In [4]:
from conn import load_connections_from_hf

hf_split = load_connections_from_hf()
print("puzzles:", len(hf_split))

c:\Users\kunri\miniconda3\envs\cs175\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


puzzles: 652


In [ ]:
#Number of times LLM can retry if it hallucinates before marking it as an error
MAX_RETRIES = 4

def valid_prediction(pred_groups, words16):
    if pred_groups is None:
        return False
    if len(pred_groups) != 4:
        return False
    if any(len(group) != 4 for group in pred_groups):
        return False
    
    pred_words = [word for group in pred_groups for word in group]
    if set(pred_words) == set(words16):
        return True
    else:
        return False

def make_few_shot_prompt(words16, k, split_for_few_shot):
    few_shot_prompt = "Here are some previous examples:\n"
    count = 0
    for fs_row in split_for_few_shot:
        fs_words = fs_row["words"]
        # Skip because of leakage
        if set(fs_words) == set(words16):
            continue
        fs_groups = [ans["words"] for ans in fs_row["answers"]]
        fs_text = f"Here are 16 words: {', '.join(fs_words)}\n"
        for i, group in enumerate(fs_groups, 1):
            fs_text += f"GROUP {i}: {', '.join(group)}\n"
        few_shot_prompt += fs_text + "\n"
        count += 1
        if count >= k:
            break
    return few_shot_prompt

def solve_puzzle(words16, k=0, split_for_few_shot=None, model="llama3.1-8b", temperature=0.1, max_tokens=600, max_retries=MAX_RETRIES):
    few_shot_prompt = ""
    if k > 0 and split_for_few_shot is not None:
        few_shot_prompt = make_few_shot_prompt(words16, k, split_for_few_shot)
    user_prompt = few_shot_prompt + "NOW, solve this puzzle and only this one: " + convert_puzzle_to_prompt(words16)

    attempt = 0
    while attempt <= max_retries:
        response = call_llm(
            SYSTEM_PROMPT_STRING,
            user_prompt,
            model=model,
            temperature=(temperature + 0.1*attempt),  
            max_tokens=max_tokens
        )

        pred_groups = parse_response_to_pred_groups(response)

        if valid_prediction(pred_groups, words16):
            return pred_groups

        print(f"Invalid LLM output: {pred_groups}\nRetrying ({attempt+1}/{max_retries})")
        attempt += 1

    print("ERROR: LLM failed after retries")
    return []

In [6]:
# Pick the first puzzle
row0 = hf_split[6]
words16 = row0["words"]

# Solve it with zero few-shot examples
pred_groups = solve_puzzle(words16, k=10)

# Display predicted groups
print("Predicted groups:")
for i, g in enumerate(pred_groups, 1):
    print(f"GROUP {i}: {g}")

# Display gold groups for comparison
print("\nGold groups:")
for ans in row0["answers"]:
    print(ans["answerDescription"], "->", ans["words"])

Predicted groups:
GROUP 1: ['EASY', 'FLEXIBLE', 'OPEN', 'RECEPTIVE']
GROUP 2: ['EVIL', 'LIVE', 'VEIL', 'VILE']
GROUP 3: ['AMAZING', 'BEGINNER', 'GENIUS', 'SOLID']
GROUP 4: ['LIT', 'SCENTED', 'WAXY', 'WICKED']

Gold groups:
AMENABLE -> ['EASY', 'FLEXIBLE', 'OPEN', 'RECEPTIVE']
ANAGRAMS -> ['EVIL', 'LIVE', 'VEIL', 'VILE']
SPELLING BEE RANKS -> ['AMAZING', 'BEGINNER', 'GENIUS', 'SOLID']
ADJECTIVES FOR A CANDLE -> ['LIT', 'SCENTED', 'WAXY', 'WICKED']


In [ ]:
from conn import (
    accuracy_min_swaps,
    accuracy_zero_one,
    evaluate,
    gold_groups_from_row,
)

N_EVAL = 100

few_shot_values = [0, 1, 5, 10, 20]
for k in few_shot_values:
    #Split to avoid leakage
    indices = list(range(k, len(hf_split)))
    split_subset = hf_split.select(indices)
    few_shot_subset = hf_split.select(list(range(k)))

    results = evaluate(
        split_subset,  #use only puzzles after the first k
        metric_fns={
            "zero_one": accuracy_zero_one,
            "min_swaps": accuracy_min_swaps,
        },
        solver_fn=lambda words: solve_puzzle(words, k=k, split_for_few_shot=few_shot_subset),
        max_samples=N_EVAL,
        gold_from_row=gold_groups_from_row,
        verbose=True,
    )
    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]

    print(f"k={k} | Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
    print(f"k={k} | Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")

Processed 2/100 samples
Processed 4/100 samples
Processed 6/100 samples
Processed 8/100 samples
Processed 10/100 samples
Processed 12/100 samples
Processed 14/100 samples
Processed 16/100 samples
Processed 18/100 samples
Processed 20/100 samples
Processed 22/100 samples
Processed 24/100 samples
Invalid LLM output, retrying (1/3)
Processed 26/100 samples
Invalid LLM output, retrying (1/3)
Invalid LLM output, retrying (2/3)
Invalid LLM output, retrying (3/3)
Processed 28/100 samples
Processed 30/100 samples
Processed 32/100 samples
Processed 34/100 samples
Invalid LLM output, retrying (1/3)
Invalid LLM output, retrying (2/3)
Processed 36/100 samples
Processed 38/100 samples
Invalid LLM output, retrying (1/3)
Processed 40/100 samples
Processed 42/100 samples
Processed 44/100 samples
Processed 46/100 samples
Processed 48/100 samples
Invalid LLM output, retrying (1/3)
Invalid LLM output, retrying (2/3)
Invalid LLM output, retrying (3/3)
Invalid LLM output, retrying (4/3)
ERROR: LLM failed a